# Phase 2: Data Preprocessing and Splitting

In this notebook, we parse the raw McMiner benchmark, analyze it for quality and leakage, and create clean, split datasets for ML modeling.

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split

RAW_DIR = '../data/raw/mcminer/corrupted_codes_best'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Step 1: Parse the dataset

In [ ]:
code_files = [f for f in glob.glob(os.path.join(RAW_DIR, '*.json')) if not f.endswith('filtering_report.json')]
print(f"Found {len(code_files)} JSON files.")

records = []
for cf in code_files:
    with open(cf, 'r', encoding='utf-8') as f:
        data = json.load(f)
        if 'solutions' not in data:
            continue
            
        prob_id = data.get('problem_id')
        misc_id = data.get('misconception_id') # The true 1-67 label
        
        for sol in data['solutions']:
            feedback = sol.get('feedback_loop', {})
            raw_response = feedback.get('raw_response', '')
            
            # Filter based on benchmark's evaluation of generated misconception
            if '<exhibits_misconception>N</exhibits_misconception>' in raw_response:
                continue
                
            record = {
                'example_id': f"{prob_id}_{sol.get('solution_index', '')}_{misc_id}",
                'problem_id': prob_id,
                'generated_code': sol.get('generated_code', ''),
                'global_misconception_index': misc_id,
                'problem_misconception_index': sol.get('problem_misconception_index'),
                'feedback_loop': json.dumps(feedback),
                'reasoning': sol.get('reasoning', ''),
                'is_compilable': feedback.get('parse_success', True)
            }
            records.append(record)

df = pd.DataFrame(records)
print(f"Total parsed examples: {len(df)}")
df.to_csv(os.path.join(PROCESSED_DIR, 'mcminer_all.csv'), index=False)

## Step 2: Data Quality Analysis

In [ ]:
missing_code = df['generated_code'].isnull().sum() + (df['generated_code'] == '').sum()
missing_labels = df['global_misconception_index'].isnull().sum()
exact_duplicates = df.duplicated().sum()
code_duplicates = df.duplicated(subset=['generated_code']).sum()

print(f"Missing code: {missing_code}")
print(f"Missing labels: {missing_labels}")
print(f"Exact duplicate records: {exact_duplicates}")
print(f"Duplicate code (regardless of label): {code_duplicates}")

print("\nCompilability distribution:")
print(df['is_compilable'].value_counts(dropna=False))

invalid_records = df[df['generated_code'].str.strip() == '']
print(f"\nMalformed/Invalid records: {len(invalid_records)}")

# Handle duplicates by keeping the first occurrence
df_clean = df.drop_duplicates(subset=['generated_code']).copy()
print(f"Remaining examples after deduplication: {len(df_clean)}")

## Step 3 & 4: Leakage Investigation
We drop `reasoning` from features for modeling because it explicitly identifies the misconception. We also must ensure that the same exact code doesn't span across splits, which we just handled via deduplication.
Furthermore, we should see if a `problem_id` heavily biases certain classes, but since we are doing misconception detection *given* a problem, stratifying strictly on the target class `global_misconception_index` is typically sufficient as long as identical code isn't duplicated.

In [ ]:
class_counts = df_clean['global_misconception_index'].value_counts()
print(f"Number of classes: {len(class_counts)}")
rare_classes = class_counts[class_counts < 10]
print(f"Classes with fewer than 10 examples: {len(rare_classes)} classes.")
if len(rare_classes) > 0:
    print("Rare class IDs:", rare_classes.index.tolist())

## Step 5: Train / Validation / Test Splits

Due to the small dataset size (and extreme class imbalance), a standard stratified split may fail if a class has too few instances to be divided properly into an 80/10/10 ratio. We will map classes with < 10 instances exclusively into the training set to preserve them, and then stratify the remainder.

In [ ]:
random_seed = 42

# Isolate rare classes to avoid split errors
rare_idx = df_clean['global_misconception_index'].isin(rare_classes.index)
df_rare = df_clean[rare_idx].copy()
df_main = df_clean[~rare_idx].copy()

# We will do 80/10/10 on the main set
train_main, temp_main = train_test_split(
    df_main, 
    test_size=0.2, 
    stratify=df_main['global_misconception_index'],
    random_state=random_seed
)

val_main, test_main = train_test_split(
    temp_main, 
    test_size=0.5, 
    stratify=temp_main['global_misconception_index'],
    random_state=random_seed
)

# Combine rare classes back into training
df_train = pd.concat([train_main, df_rare]).sample(frac=1, random_state=random_seed).reset_index(drop=True)
df_val = val_main.reset_index(drop=True)
df_test = test_main.reset_index(drop=True)

print(f"Final Train size: {len(df_train)}")
print(f"Final Validation size: {len(df_val)}")
print(f"Final Test size: {len(df_test)}")

df_train.to_csv(os.path.join(PROCESSED_DIR, 'train.csv'), index=False)
df_val.to_csv(os.path.join(PROCESSED_DIR, 'validation.csv'), index=False)
df_test.to_csv(os.path.join(PROCESSED_DIR, 'test.csv'), index=False)

print("Splits successfully created and saved!")

## 6. Visualization

In [ ]:
plt.figure(figsize=(15, 6))
df_train['global_misconception_index'].value_counts().plot(kind='bar', color='blue', alpha=0.6, label='Train')
df_val['global_misconception_index'].value_counts().plot(kind='bar', color='orange', alpha=0.6, label='Val')
df_test['global_misconception_index'].value_counts().plot(kind='bar', color='green', alpha=0.6, label='Test')
plt.title('Misconception Class Distribution Across Splits')
plt.xlabel('Misconception Index')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'class_distribution.png'))
plt.show()